# 01 Single Image Quickstart

Directive notebook for:
- setup/install
- loading one image
- extracting classifier-head principal projector
- translating with `resnet` and `dinovit1`
- computing `AS` (classes + attributes vectors) and `IS`
- printing and plotting results

No core metric/projector logic is defined here; the notebook only calls package APIs.

In [ ]:
#---------------------------- setup install --------------------------------#
# Local (non-Docker) recommended flow: create venv in terminal, then reopen notebook with that kernel.
# python -m venv ../.venv
# source ../.venv/bin/activate
# pip install -r ../requirements.txt
# pip install -e ..

# Google Colab / notebook-only fallback:
# %pip install -r ../requirements.txt
# %pip install -e ..

In [ ]:
#---------------------------- gpu check --------------------------------#
!nvidia-smi

In [ ]:
#---------------------------- imports and config --------------------------------#
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

from sing.core.metrics import compute_as, compute_is
from sing.core.projectors import compute_projectors_from_weight, principal_component
from sing.models.registry import load_model
from sing.translators.registry import load_default_translator

repo_root = Path("..").resolve()
image_path = repo_root / "samples" / "goldfish_n01443537.jpeg"
if not image_path.exists():
    raise FileNotFoundError(f"Set image_path to an existing image. Missing: {image_path}")

# Plot the image from samples
image = Image.open(image_path).convert("RGB")
plt.figure(figsize=(4, 4))
plt.imshow(image)
plt.axis('off')
plt.title("Input Image")
plt.show()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={device}")

In [ ]:
#---------------------------- load npz embeddings --------------------------------#
classes_npz = np.load(repo_root / "misc" / "embeddings" / "imagenet_text_embeddings_fp16.npz", allow_pickle=True)
attrs_npz = np.load(repo_root / "misc" / "embeddings" / "attribute_embeddings_broden_style_fp16.npz", allow_pickle=True)

class_labels = classes_npz["classes"].astype(str)
attr_labels = attrs_npz["classes"].astype(str)

class_prompt_names = classes_npz["prompts"].astype(str)
attr_prompt_names = attrs_npz["prompts"].astype(str)

class_prompt_idx = int(np.where(class_prompt_names == "mean")[0][0]) if (class_prompt_names == "mean").any() else 0
attr_prompt_idx = int(np.where(attr_prompt_names == "mean")[0][0]) if (attr_prompt_names == "mean").any() else 0

class_text_embeddings = torch.tensor(classes_npz["embeddings"][:, class_prompt_idx, :], dtype=torch.float32, device=device)
attr_text_embeddings = torch.tensor(attrs_npz["embeddings"][:, attr_prompt_idx, :], dtype=torch.float32, device=device)

print(f"class_text_embeddings={tuple(class_text_embeddings.shape)}")
print(f"attr_text_embeddings={tuple(attr_text_embeddings.shape)}")

In [ ]:
#---------------------------- run resnet and dinovit1 --------------------------------#
image = Image.open(image_path).convert("RGB")

results = {}
for model_name in ["resnet", "dinovit1"]:
    loaded_model = load_model(model_name=model_name, device=device)
    loaded_translator = load_default_translator(
        translators_root=repo_root / "translators",
        registry_path=repo_root / "translators" / "registry.yaml",
        model_name=loaded_model.name,
        device=device,
    )

    input_tensor = loaded_model.preprocess(image).unsqueeze(0).to(device)

    with torch.no_grad():
        features = loaded_model.wrapper.extract_features(input_tensor)
        classifier_weight = loaded_model.wrapper.classifier_weight.detach().to(device)
        projectors = compute_projectors_from_weight(classifier_weight)
        principal_features = principal_component(features, projectors.v_null)

        translated_original = loaded_translator.model(features)
        translated_principal = loaded_translator.model(principal_features)

        is_value = float(compute_is(translated_original, translated_principal).mean().item())
        as_classes = compute_as(translated_original, translated_principal, class_text_embeddings)
        as_attributes = compute_as(translated_original, translated_principal, attr_text_embeddings)

    results[model_name] = {
        "is": is_value,
        "as_classes": as_classes.detach().cpu(),
        "as_attributes": as_attributes.detach().cpu(),
        "rank": int(projectors.rank),
        "v_null_shape": tuple(projectors.v_null.shape),
        "v_principal_shape": tuple(projectors.v_principal.shape),
    }

In [ ]:
#---------------------------- print metrics --------------------------------#
for model_name in ["resnet", "dinovit1"]:
    r = results[model_name]
    print(f"\nmodel={model_name}")
    print(f"IS={r['is']:.6f}")
    print(f"projector_rank={r['rank']} v_null_shape={r['v_null_shape']} v_principal_shape={r['v_principal_shape']}")
    print(f"AS_classes_vector_shape={tuple(r['as_classes'].shape)}")
    print(f"AS_attributes_vector_shape={tuple(r['as_attributes'].shape)}")

In [ ]:
#---------------------------- plot metrics --------------------------------#
fig, axes = plt.subplots(2, 2, figsize=(20, 12))
models = ["resnet", "dinovit1"]

# Compute overall y-limits for all bars (classes and attributes), then plot with shared y-lim
max_y = 0.0
for model_name in models:
    r = results[model_name]
    as_classes = r["as_classes"].numpy()
    as_attrs = r["as_attributes"].numpy()
    max_y = max(
        max_y,
        np.abs(as_classes).max() if as_classes.size else 0,
        np.abs(as_attrs).max() if as_attrs.size else 0,
    )

fontdict = {'fontsize': 24}

for col, model_name in enumerate(models):
    r = results[model_name]

    as_classes = r["as_classes"].numpy()
    as_attrs = r["as_attributes"].numpy()

    topk_c = min(20, as_classes.shape[0])
    topk_a = min(20, as_attrs.shape[0])

    idx_c = np.argsort(np.abs(as_classes))[-topk_c:]
    idx_a = np.argsort(np.abs(as_attrs))[-topk_a:]

    order_c = idx_c[np.argsort(np.abs(as_classes[idx_c]))[::-1]]
    order_a = idx_a[np.argsort(np.abs(as_attrs[idx_a]))[::-1]]

    class_vals = np.abs(as_classes[order_c])
    attr_vals = np.abs(as_attrs[order_a])

    class_names = class_labels[order_c]
    attr_names = attr_labels[order_a]

    # Plot classes
    axes[0, col].bar(np.arange(topk_c), class_vals)
    axes[0, col].set_title(f"{model_name} | top-{topk_c} |AS| classes", fontsize=24)
    axes[0, col].set_ylabel("AS (deg)", fontsize=24)
    axes[0, col].set_ylim(0, max_y * 1.05)
    axes[0, col].set_xticks(np.arange(topk_c))
    axes[0, col].set_xticklabels(class_names, rotation=60, ha="right", fontsize=24)
    axes[0, col].tick_params(axis='x', labelsize=24)
    axes[0, col].tick_params(axis='y', labelsize=24)
    # Plot attributes
    axes[1, col].bar(np.arange(topk_a), attr_vals, color="tab:orange")
    axes[1, col].set_title(f"{model_name} | top-{topk_a} |AS| attributes", fontsize=24)
    axes[1, col].set_ylabel("AS (deg)", fontsize=24)
    axes[1, col].set_ylim(0, max_y * 1.05)
    axes[1, col].set_xticks(np.arange(topk_a))
    axes[1, col].set_xticklabels(attr_names, rotation=60, ha="right", fontsize=24)
    axes[1, col].tick_params(axis='x', labelsize=24)
    axes[1, col].tick_params(axis='y', labelsize=24)


plot_path = repo_root / "outputs" / "notebook_single_image_metrics.png"
plot_path.parent.mkdir(parents=True, exist_ok=True)
plt.tight_layout()
plt.savefig(plot_path, dpi=180, bbox_inches="tight")
print(f"saved plot: {plot_path}")
plt.show()